# Classification NBA Model

## Configuration

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# from nba_ou.data_processing.missing_data.handle_missing_data import (
#     apply_missing_policy,
# )
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold, cross_validate, train_test_split


## Load Data

In [ ]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260120.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [ ]:
# #print column names with name in lower case
# for col in df_stats.columns:
#     if not col.isupper():
#         print(col)

In [ ]:
#filter for last two seasons, gett seasons 2025 and 2024
# df_stats = df_stats[df_stats['SEASON_YEAR'].isin([2024, 2025])]

In [ ]:
df_stats

In [ ]:
from nba_ou.data_processing.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=3, drop_all_na_rows=True, verbose=1)


In [ ]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_stats[df_stats[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_stats.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

In [ ]:
df_to_train['LINE_ERROR'] = df_to_train['TOTAL_POINTS'] - df_to_train['TOTAL_OVER_UNDER_LINE']

In [ ]:
# df_to_train = df_to_train[df_to_train['LINE_ERROR'] != 0]

## Train / Test

In [ ]:
X = df_to_train.drop(['TOTAL_POINTS', 'LINE_ERROR', 'SEASON_YEAR'], axis=1, errors='ignore')
y = df_to_train['LINE_ERROR']

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=16)

In [ ]:
df_to_train['IS_TRAINING_DATA'] = False

# Mark True for the rows in the training set
df_to_train.loc[X_train.index, 'IS_TRAINING_DATA'] = True
# output_name = f"{data_path}/training_data_with_missing_data_handled_from_2004-10-01_to_2026-01-10_classifier.csv"
# df_to_train.to_csv(output_name, index=False)

In [ ]:
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
# Check number of coulmns
print(f"Number of columns in training set: {X_train.shape[1]}")
print(f"Number of columns in test set: {X_test.shape[1]}")

In [ ]:
from tabpfn_client import init, TabPFNRegressor
# Load an example dataset


# Use it like any sklearn model
model = TabPFNRegressor()
model.fit(X_train, y_train)
# Get predictions
predictions = model.predict(X_test)
# Get probability estimates


In [ ]:
from tabpfn.model_loading import (
    load_fitted_tabpfn_model,
    save_fitted_tabpfn_model,
)

save_fitted_tabpfn_model(model, "my_reg.tabpfn_fit")

# Later or on a CPU-only machine
reg_cpu = load_fitted_tabpfn_model("my_reg.tabpfn_fit", device="cpu")

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
# Predict on the test set

def over_under_betting_accuracy(true_error: np.ndarray, pred_error: np.ndarray) -> float:
    true_sign = np.sign(true_error)
    pred_sign = np.sign(pred_error)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))

# Evaluate the model
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
betting_acc = over_under_betting_accuracy(y_test.values, predictions)

print("Mean Squared Error (MSE):", mse)
print("R² Score:", r2)
print("Mean Absolute Error (MAE):", mae)
print("Over/Under Betting Accuracy:", betting_acc)

In [ ]:
import numpy as np
import pandas as pd


def error_sign_accuracy(
    y_true_error,
    y_pred_error,
    include_pushes: bool = False,
) -> float:
    """
    Directional accuracy: sign(true_error) == sign(pred_error).

    Push handling:
    - Default: exclude any sample where either sign == 0
    - If include_pushes=True: count (0,0) as correct, exclude mixed 0/non-0
    """
    y_true_error = np.asarray(y_true_error, dtype=float)
    y_pred_error = np.asarray(y_pred_error, dtype=float)

    true_sign = np.sign(y_true_error)
    pred_sign = np.sign(y_pred_error)

    if include_pushes:
        valid = ~((true_sign == 0) ^ (pred_sign == 0))
    else:
        valid = (true_sign != 0) & (pred_sign != 0)

    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))

def evaluate_by_confidence_quantiles(
    y_true_error: pd.Series,
    y_pred_error: np.ndarray,
    quantiles=(0.05, 0.10, 0.20, 0.30, 0.50, 1.0),
    include_pushes: bool = False,
) -> pd.DataFrame:
    """
    Directional accuracy on the top-q most confident predictions,
    where confidence = abs(pred_error).
    """
    y_true = y_true_error.to_numpy(dtype=float)
    y_pred = np.asarray(y_pred_error, dtype=float)

    abs_pred = np.abs(y_pred)
    n_total = len(abs_pred)

    order = np.argsort(-abs_pred)

    rows = []
    for q in quantiles:
        k = int(np.ceil(q * n_total))
        idx = order[:k]

        acc = (
            np.nan
            if k == 0
            else error_sign_accuracy(
                y_true[idx],
                y_pred[idx],
                include_pushes=include_pushes,
            )
        )

        rows.append({
            "top_confidence_fraction": float(q),
            "n_games": k,
            "pct_of_test": (k / n_total) if n_total else np.nan,
            "directional_accuracy": acc,
            "min_abs_pred_error": float(abs_pred[idx].min()) if k else np.nan,
            "mean_abs_pred_error": float(abs_pred[idx].mean()) if k else np.nan,
        })

    return pd.DataFrame(rows)

def evaluate_by_abs_pred_error_thresholds(
    y_true_error: pd.Series,
    y_pred_error: np.ndarray,
    thresholds = np.arange(0.0, 11.0, 0.5),
    include_pushes: bool = False,
) -> pd.DataFrame:
    """
    Directional accuracy conditional on abs(pred_error) > t.
    """
    y_true = y_true_error.to_numpy(dtype=float)
    y_pred = np.asarray(y_pred_error, dtype=float)

    abs_pred = np.abs(y_pred)
    n_total = len(y_true)

    rows = []
    for t in thresholds:
        mask = abs_pred > t
        n = int(mask.sum())

        acc = (
            np.nan
            if n == 0
            else error_sign_accuracy(
                y_true[mask],
                y_pred[mask],
                include_pushes=include_pushes,
            )
        )

        rows.append({
            "abs_pred_error_gt": float(t),
            "n_games": n,
            "pct_of_test": (n / n_total) if n_total else np.nan,
            "directional_accuracy": acc,
            "mean_abs_pred_error": float(abs_pred[mask].mean()) if n else np.nan,
        })

    return pd.DataFrame(rows)


In [ ]:
# Threshold-based evaluation
thr_df = evaluate_by_abs_pred_error_thresholds(
    y_true_error=y_test,
    y_pred_error=predictions,
    thresholds=np.arange(0.0, 5.0, 0.2),
)

display(
    thr_df.style.format({
        "pct_of_test": "{:.1%}",
        "directional_accuracy": "{:.2%}",
        "mean_abs_pred_error": "{:.2f}",
    })
)

# Quantile-based evaluation
quant_df = evaluate_by_confidence_quantiles(
    y_true_error=y_test,
    y_pred_error=predictions,
)

display(
    quant_df.style.format({
        "pct_of_test": "{:.1%}",
        "directional_accuracy": "{:.2%}",
        "mean_abs_pred_error": "{:.2f}",
    })
)
